<a href="https://colab.research.google.com/github/harry934/Emergency-Responce/blob/colab-code/EMERGENCY_RESPONSE_IMAGE_PRETRAINED_MODEL_MobileNETV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

EMERGENCY RESPONSE IMAGE CLASSIFICATION USING CNN

Student Names:

1. HARRY MOKAYA - 669412

2. ELIJAH LEMPOKO -

3.


Course Name : SWE 2020A

Supervisor Name : EDWARD OMBUI

## **INTRODUCTION**
Emergency response systems rely heavily on early detection, quick assessment, and accurate classification of scenes such as fires, accidents, and medical emergencies. This project implements a CNN model designed to automatically classify emergency-related images to support faster decision-making.

This project also builds a machine learning model capable of classifying:

- **Accidents**
- **Heavy Traffic**
- **Normal Road Activity**

The objective of this work is to:
- Build an end-to-end emergency image classification system  
- Perform image preprocessing and augmentation  
- Train a CNN model on emergency scene images  
- Evaluate model performance using standard metrics  
- Present an interface suitable for real-world use  

The system further integrates a **dispatch algorithm** that identifies the nearest emergency unit based on coordinates.  

The full pipeline includes:

1. Dataset Loading.
2. Exploratory Data Analysis.
3. Image Preprocessing.
4. Image Augmentation.
5. Model Architecture.
6. Model Training.
7. Model Evaluation.
8. Conclusion.
9. User Interface Prototype



## **DATASET DESCRIPTION:**

The dataset consists of three classes:

1. **Accident**
2. **HeavyTraffic**
3. **NormalRoadActivity**

Each image is resized, normalized to **0–1**, and prepared using Keras ImageDataGenerator.

The Important dataset steps:

- Extraction from a ZIP file  
- Identification of class distribution  
- Removal of corrupted images  
- Balancing of classes via horizontal flipping  


SECTION 1 - DATASET UPLOAD & EXTRACTION

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import zipfile
from tensorflow.keras.preprocessing.image import load_img, img_to_array, ImageDataGenerator
from google.colab import files
from tensorflow.keras.preprocessing import image
import numpy as np

print("Upload an image to test the model prediction:")
uploaded = files.upload()

if not os.path.exists('Dataset'):
    with zipfile.ZipFile('Dataset.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
    print("Dataset extracted successfully.")


Upload an image to test the model prediction:


SECTION 2: IMAGE GENERATORS

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_data = train_datagen.flow_from_directory(
    "Dataset",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_data = val_datagen.flow_from_directory(
    "Dataset",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)


SECTON 3: EXPLORATORY DATA ANALYSIS (EDA)

In [ ]:
classes = list(train_data.class_indices.keys())
print("Classes found:", classes)

counts = []
for cls in classes:
    path = os.path.join("Dataset", cls)
    count = len(os.listdir(path))
    counts.append(count)
    print(f"{cls}: {count} images")

plt.figure(figsize=(8,5))
bars = plt.bar(classes, counts, color=['#FF6F61', '#6B5B95', '#88B04B'])
plt.title("Class Distribution in Dataset", fontsize=16, fontweight='bold')
plt.xlabel("Class", fontsize=12)
plt.ylabel("Number of Images", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)


for bar, count in zip(bars, counts):
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 2, count, ha='center', va='bottom', fontsize=11)

plt.show()


def plot_samples(dataset_path, classes, img_size=(224,224), n_samples=3):
    plt.figure(figsize=(12, 6))
    for i, cls in enumerate(classes):
        cls_path = os.path.join(dataset_path, cls)
        images = os.listdir(cls_path)[:n_samples]
        for j, img_file in enumerate(images):
            img = load_img(os.path.join(cls_path, img_file), target_size=img_size)
            ax = plt.subplot(len(classes), n_samples, i*n_samples + j + 1)
            plt.imshow(img)
            plt.axis('off')
            if j == 1:
                ax.set_title(cls, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_samples("Dataset", classes, IMG_SIZE)

SECTION 4: CLEANING THE IMAGES

In [ ]:
from PIL import Image

for cls in classes:
    cls_path = os.path.join("Dataset", cls)
    for img_file in os.listdir(cls_path):
        img_path = os.path.join(cls_path, img_file)
        try:
            img = Image.open(img_path)
            img.verify()
        except:
            print("Removing corrupted image:", img_path)
            os.remove(img_path)


## **Image Augmentation**

To improve model generalization, the following augmentation transformations are applied:

- Rotation (20°)
- Width and height shifting
- Shearing
- Zooming
- Horizontal flipping
- Fill mode for missing pixels

These techniques will reduce overfitting and help the model adapt to different conditions.



SECTION 5: BALANCE DATASET BY FLIPPING

In [ ]:
import cv2
import random
import os

def flip_images_in_folder(folder, target_count):
    files = [f for f in os.listdir(folder) if os.path.isfile(os.path.join(folder, f))]
    while len(files) < target_count:
        if not files:
            break
        img_name = random.choice(files)
        img_path = os.path.join(folder, img_name)
        img = cv2.imread(img_path)
        if img is None:
            files.remove(img_name)
            continue
        flipped = cv2.flip(img, 1)
        new_name = f"aug_{len(os.listdir(folder))}_{random.randint(0, 1000)}.jpg"
        cv2.imwrite(os.path.join(folder, new_name), flipped)
        files = [f for f in os.listdir(folder) if os.path.isfile(os.path.join(folder, f))]

max_count = max(counts)

for i, cls_name in enumerate(classes):
    folder = os.path.join("Dataset", cls_name)
    if counts[i] < max_count:
        flip_images_in_folder(folder, max_count)

print("Dataset balanced successfully.")


## **EDA REPORT**
- Classes and Samples: The dataset contains 3 classes (Accident, HeavyTraffic, NormalRoadActivity).
The number of images per class varies slightly, but all classes are adequately represented.

- Sample Images: Visual inspection of sample images shows differences in lighting, camera angles, and resolutions.
This justifies the use of augmentation techniques like flipping, rotation, and zooming.

- Image Quality: Some images are low-quality or corrupted. These images were removed to ensure consistent training data.


**Insights:**

- Dataset is moderately balanced.

- Augmentation improved model generalization.

- Preprocessing (resizing, normalization) was necessary due to varied image sizes and lighting conditions.



## **Model Training**

MobileNetV2 pretrained on ImageNet is used as the feature extractor.  
Two training stages were performed:

### **Stage 1 — Freeze Base Model**
Only custom layers train.

### **Stage 2 — Fine-Tuning**
Entire MobileNetV2 unfreezes at a low learning rate (1e-5) for improved accuracy.


SECTION 6: BUILD & TRAIN MODEL USING TRANSFER LEARNING MOBILENETV2

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model

# Stage 1 — Freeze Base Model
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)


base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)

output = Dense(3, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(train_data, validation_data=val_data, epochs=10)


SECTION 7: FINE-TUNE MODEL

In [ ]:
# Stage 2 — Fine-Tuning

base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(train_data, validation_data=val_data, epochs=5)


SECTION 8: SAVE MODEL

In [ ]:
model.save("emergency_cnn_model.keras")
print("Model saved as emergency_cnn_model.keras")


## **Model Evaluation**

The model performance is evaluated using:

### **Training & Validation Accuracy**
Indicates how well the model fits the dataset.

### **Training & Validation Loss**
Measures model error over epochs.

### **Confusion Matrix**
Shows the distribution of correct and incorrect predictions.

### **Classification Report**
Includes:
- Precision  
- Recall  
- F1-score  
- Support  



In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

val_data.reset()
predictions = model.predict(val_data)
predicted_labels = np.argmax(predictions, axis=1)
true_labels = val_data.classes

print("Classification Report:")
print(classification_report(true_labels, predicted_labels, target_names=classes))

cm = confusion_matrix(true_labels, predicted_labels)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=classes, yticklabels=classes)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## **Conclusion**

The developed system successfully classifies road conditions into:

- Accident
- Heavy Traffic
- Normal Road Activity

The MobileNetV2 model demonstrated strong performance based on precision, recall, F1-score, and confusion matrix results.

The system also integrates a real-time emergency dispatch module that identifies the nearest emergency unit using geospatial distance.

This model can be deployed in:
- Traffic monitoring systems  
- Smart city infrastructure  
- Real-time emergency alert systems  


## **Prediction Interface**

The user uploads an image and receives:

1. Classification result  
2. Confidence values  
3. Nearest emergency unit (if accident detected)  
4. Full dispatch summary  


SECTION 9: UPLOAD IMAGE FOR PREDICTION

In [ ]:
from google.colab import files
from tensorflow.keras.preprocessing import image
import numpy as np

print("Upload an image to classify (JPG/PNG):")
uploaded_test = files.upload()


SECTION 10:PREDICT AND SHOW CONFIDENCE

In [ ]:
import matplotlib.pyplot as plt

labels = ["Accident", "HeavyTraffic", "NormalRoadActivity"]

for fn in uploaded_test.keys():
    img = image.load_img(fn, target_size=IMG_SIZE)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)[0]
    final_decision = labels[np.argmax(prediction)]

    print("\nIMAGE:", fn)
    print("Raw Prediction:", prediction)
    print("Final Decision:", final_decision)

    plt.figure(figsize=(6,4))
    plt.bar(labels, prediction)
    plt.title("Prediction Confidence")
    plt.ylabel("Probability")
    plt.ylim(0, 1)
    plt.show()


SECTION 11:NEAREST EMERGENCY UNIT DISBATCH

In [ ]:
from math import radians, sin, cos, asin, sqrt
from datetime import datetime

emergency_units = [
    {"name": "Nairobi West Hospital", "lat": -1.3035, "lon": 36.7891},
    {"name": "KNH", "lat": -1.2921, "lon": 36.8219},
    {"name": "Mbagathi Hospital", "lat": -1.3540, "lon": 36.8140}
]

camera_lat = -1.2921
camera_lon = 36.8219

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2 * R * asin(sqrt(a))

if final_decision != "Accident":
    nearest = None
    shortest_dist = None
    print("\nNo ambulance dispatch needed. Prediction was:", final_decision)
else:
    nearest = None
    shortest_dist = 9999
    for unit in emergency_units:
        dist = haversine(camera_lat, camera_lon, unit["lat"], unit["lon"])
        if dist < shortest_dist:
            shortest_dist = dist
            nearest = unit

    print("\nEMERGENCY DETECTED: Accident!")
    print("Nearest emergency unit:", nearest["name"])
    print("Distance (km):", round(shortest_dist, 2))


SECTION 12: ALERT SUMMARY

In [ ]:
if final_decision == "Accident":
    alert = {
        "image_tested": list(uploaded_test.keys())[0],
        "prediction": final_decision,
        "confidence": float(max(prediction)),
        "camera_location": {"lat": camera_lat, "lon": camera_lon},
        "nearest_emergency_unit": nearest["name"],
        "distance_km": round(shortest_dist, 2),
        "alert_timestamp": datetime.utcnow().isoformat()
    }

    print("\n================ EMERGENCY ALERT ================")
    print(alert)
    print("================================================")

else:
    alert = {
        "image_tested": list(uploaded_test.keys())[0],
        "prediction": final_decision,
        "confidence": float(max(prediction)),
        "message": "No emergency dispatch needed.",
        "alert_timestamp": datetime.utcnow().isoformat()
    }

    print("\n================ NO ALERT ================")
    print(alert)
    print("============================================")
